# Lag-Llama: Лаги как входы для вероятностного прогнозирования

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/35_lag_llama.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q gluonts torch pandas numpy matplotlib huggingface_hub
!pip install -q lag-llama

## Подготовка данных

In [ ]:
import torch
import pandas as pd
import numpy as np
from gluonts.dataset.pandas import PandasDataset

# Создаём синтетические данные с сезонностью
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')

# Несколько рядов
series_data = []
for i in range(5):
    y = 100 + np.cumsum(np.random.randn(365)) + 20 * np.sin(np.arange(365) / 7 * 2 * np.pi)
    series_data.append(pd.DataFrame({
        'unique_id': f'series_{i}',
        'ds': dates,
        'y': y
    }))

df = pd.concat(series_data, ignore_index=True)

# Создаём GluonTS датасет
dataset = PandasDataset.from_long_dataframe(
    df,
    item_id='unique_id',
    timestamp='ds',
    target='y',
    freq='D'
)

print(f"Количество рядов: {df['unique_id'].nunique()}")

## Lag-Llama: загрузка модели

In [ ]:
from huggingface_hub import hf_hub_download
from lag_llama.gluon.estimator import LagLlamaEstimator

# Загружаем предобученную модель
model_path = hf_hub_download(
    repo_id="time-series-foundation-models/Lag-Llama",
    filename="lag-llama.ckpt"
)

print(f"Модель загружена: {model_path}")

In [ ]:
# Создаём estimator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

estimator = LagLlamaEstimator(
    ckpt_path=model_path,
    prediction_length=24,      # горизонт прогноза
    context_length=96,         # длина контекста
    num_samples=100,           # сэмплов для вероятностного прогноза
    device=device,
)

# Создаём predictor
predictor = estimator.create_predictor(
    estimator.create_transformation(),
    estimator.create_lightning_module()
)

print(f"Predictor создан на устройстве: {device}")

## Генерация прогнозов

In [ ]:
# Генерируем прогнозы
forecasts = list(predictor.predict(dataset))

print(f"Количество прогнозов: {len(forecasts)}")

# Получаем медиану и интервалы для первого прогноза
for i, forecast in enumerate(forecasts[:3]):
    median = forecast.median
    lower_90 = forecast.quantile(0.05)
    upper_90 = forecast.quantile(0.95)
    print(f"\nSeries {i}:")
    print(f"  Median (first 5): {median[:5]}")
    print(f"  90% interval width: {(upper_90 - lower_90).mean():.2f}")

## Визуализация вероятностного прогноза

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ax in enumerate(axes[:4]):
    # История
    series = df[df['unique_id'] == f'series_{i}']
    history = series.tail(50)
    
    ax.plot(history['ds'], history['y'], 'b-', label='История')
    
    # Прогноз
    forecast = forecasts[i]
    forecast_dates = pd.date_range(
        start=history['ds'].iloc[-1] + pd.Timedelta(days=1),
        periods=len(forecast.median),
        freq='D'
    )
    
    # Интервалы
    ax.fill_between(
        forecast_dates,
        forecast.quantile(0.05),
        forecast.quantile(0.95),
        alpha=0.2, color='red', label='90% интервал'
    )
    ax.fill_between(
        forecast_dates,
        forecast.quantile(0.25),
        forecast.quantile(0.75),
        alpha=0.4, color='red', label='50% интервал'
    )
    
    # Медиана
    ax.plot(forecast_dates, forecast.median, 'r-', linewidth=2, label='Медиана')
    
    ax.axvline(x=history['ds'].iloc[-1], color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'Series {i}')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Lag-Llama: вероятностное прогнозирование', fontsize=14)
plt.tight_layout()
plt.show()

## Лаговая последовательность Lag-Llama

In [ ]:
# Стандартная лаговая последовательность в Lag-Llama
LAGS_SEQUENCE = [
    1, 2, 3, 4, 5, 6, 7,           # ежедневные лаги (неделя)
    14, 21, 28,                     # двух-, трёх-, четырёхнедельные
    30, 60, 90,                     # месячные
    365, 730,                       # годовые
]

print("Лаговая последовательность Lag-Llama:")
for lag in LAGS_SEQUENCE:
    if lag <= 7:
        desc = f"{lag} день назад"
    elif lag <= 28:
        desc = f"{lag // 7} недель назад"
    elif lag <= 90:
        desc = f"~{lag // 30} месяц(а) назад"
    else:
        desc = f"~{lag // 365} год(а) назад"
    print(f"  Лаг {lag}: {desc}")